# 3. Select interaction and propagation schemes

Keep the specimen fixed while changing one model choice at a time. Scalar, Jones and Stokes select field representations; multislice on/off changes within-sample diffraction; Fraunhofer/RS changes only exit-to-camera propagation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/scattering_calculator').is_dir())
PAPER = ROOT / 'paper/scattering_calculator'
sys.path.insert(0, str(PAPER))
sys.path.insert(0, str(ROOT / "src"))
from scattering_calculator import simulation_pipelines as sim
import experiments as ex
import workflow as wf

def show(name):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(plt.imread(OUT / (name + '.png')))
    ax.axis('off')
    plt.show()


## Standard experiment setup

Use the same production configuration sections in every example:
**X-ray source → simulation grid → sample → magnetization → apertures →
illumination → sample propagation → detector/acquisition → beamstop → energy scan**.
Physical lengths are metres, energy is eV, and angles are radians; recipe
thickness literals are nm. See [the setup standard](../../../docs/experiment_setup.md).
Specialized plotting, reconstruction and analytic-reference settings are analysis
controls, separate from the physical experiment definition.


In [ ]:
# 1. X-RAY SOURCE — energy in eV, flux in photons/s, angles in radians
# 2. SIMULATION GRID — (y, x) pixels and physical pitch in metres
# 3. SAMPLE — recipe thickness literals are nm; slice thickness is metres
# 4. MAGNETIZATION — all physical pattern lengths are metres
# 5. APERTURES — positions are (y, x) in metres
# 6. ILLUMINATION — profile, focus and direction
# 7. SAMPLE PROPAGATION — field representation and multislice
# 8. DETECTOR / ACQUISITION — physical geometry, response and artifacts
# 9. BEAMSTOP; 10. ENERGY SCAN; analysis controls remain separate
experiment = sim.ExperimentConfig(
    xray=sim.XRayConfig(energy=778., photon_flux=1., pol="CR",
                       coherence_length=(100e-6, 100e-6)),
    simulation=sim.SimulationConfig(shape=(192,192), real_space_pixel_size=10e-9),
    sample=sim.SampleConfig(recipe="Au(300)/SiN(20)/Pt(2)/Co(10)/Pt(2)",
                            max_slice_thickness=5e-9, sample_name="Normal-incidence FTH"),
    magnetic_pattern=sim.MagneticPatternConfig(
        pattern_type_method="smooth_domains", pattern_config={"period":110e-9}),
    aperture=sim.FrontApertureConfig(aperture_method="FTH_circular", aperture_config={
        "apertures_type":["OH","RH"], "apertures_radius":[180e-9,30e-9],
        "apertures_center":[(0.,0.),(0.,600e-9)], "apertures_sigma":[0.,0.],
        "apertures_top_radius_factor":[1.,1.], "thickness_OH":300e-9,
    }),
    illumination=sim.IlluminationConfig(illumination_function="gaussian", illumination_config={
        "center":(0.,0.), "distance":0., "fwhm":2*np.sqrt(np.log(2))*650e-9,
        "alpha_beam":(0.,0.),
    }),
    propagation=sim.SamplePropagatorConfig(propagator_method="Jones", propagator_config={
        "propagate":True, "dielectric_tensor_compact":True,
    }),
    detector=sim.DetectorConfig(shape=(256,256), pixel_size=13.5e-6,
        sample_to_detector_distance=.05, detector_center=(128,128),
        detector_propagation_method="fraunhofer", use_detector_pixel_footprint=False,
        detector_pixel_footprint_samples=3, analyzer_angle=None,
        measurement_config={"exposure_time":1.,"number_frames":1,"max_counts_per_image":None},
        detector_params={"counts_per_photon":1.,"quantum_efficiency":1.,
            "readout_noise_average":0.,"readout_noise_sigma":2.,"detector_threshold":16000.,
            "noise_seed":17},
        artifacts_config={"sigma_photon":0.,"camera_seed":31,"average_hot_pixels":12.,
                          "average_cold_pixels":12.,"cosmic_rays_per_second":2.}),
    beamstop=sim.BeamstopConfig(bs_method="circular",bs_detector_distance=.01,
                              bs_config={"radius":40.5e-6}),
    energies_eV=tuple(np.arange(772.,801.)),
)
OUT = PAPER / 'results' / 'notebook_model_options'
OUT.mkdir(parents=True, exist_ok=True)


## Scalar, Jones and pure-state Stokes
The default circular input and zero linear material channel make Scalar a useful eigenmode control. Stokes shares its Jones carrier; agreement checks consistency.

In [ ]:
cases = {mode: ex.fth_case(config=experiment, mode=mode)
         for mode in ('Scalar','Jones','Stokes')}
ref = cases['Jones']['difference']
limit = abs(ref).max()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for ax, (mode, case) in zip(axes, cases.items()):
    error = np.linalg.norm(case['difference']-ref)/np.linalg.norm(ref)
    ax.imshow(case['difference'], origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(f'{mode}: relative L2 {error:.3g}')
plt.show()


## Multislice versus a projection control
`propagate=False` retains local interactions and omits transverse diffraction between slices. Compare on the same q grid and scale.

In [ ]:
projection = ex.fth_case(config=experiment.with_changes(propagation={'propagator_config':{**experiment.propagation.propagator_config,'propagate':False}}))
print('Projection / multislice relative difference:',
      np.linalg.norm(projection['difference']-ref)/np.linalg.norm(ref))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for ax, image, title in zip(axes, (ref,projection['difference']), ('Multislice','Projection control')):
    ax.imshow(image, origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(title)
plt.show()


## Switch exit-to-detector propagation
A small detector makes direct RS integration practical. Both calculations receive the same complex exit fields. These panels use separate relative scales: raw brightness is not a calibrated backend comparison. The legacy FFT and RS transverse-coordinate conventions differ; match conventions, source support, sampling and normalization before computing physical error metrics.

In [ ]:
from matplotlib.colors import LogNorm
small = replace(experiment.detector, shape=(24,24))
ff, _ = wf.project(cases['Jones'], small)
rs, _ = wf.project(cases['Jones'], replace(small, detector_propagation_method='rayleigh_sommerfeld'))
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for ax, image, title in zip(axes, (ff[0],rs[0]), ('Fraunhofer','Rayleigh–Sommerfeld')):
    ax.imshow(image/image.max(), origin='lower', norm=LogNorm(1e-6,1))
    ax.set_title(title + ' (own peak = 1)')
    ax.set_xlabel('detector column')
plt.show()
np.savez_compressed(OUT / 'propagation_options.npz', fraunhofer=ff, rayleigh_sommerfeld=rs)
